# V6' (V6-prime) FULL — Pivot full-LR + 11-node wind graph (ST-CDGM Path C+)

**Validé 5/5 par l'audit indépendant (2026-06-30).** Médiane P(battre noncausal sur ≥1 co-primaire) ≈ **75%** (V6 MVP rejeté ≈ 15%).

## Deux changements, tous deux recommandés par l'audit
1. **Stage 2 full-LR conditioning** (LE pivot) : `[y_noisy, mu_HR, baseline]` → `+ 22 champs LR`. Le vrai goulot était informationnel (toy diffusion : F1@p99 0.053 pauvre vs **0.639** full-info). Pattern CorrDiff/StormCast/Rampal 2025.
2. **Stage 1 : 11-node** (9-node + U850/V850 wind nodes, recommandation Climat) + 21 LR drivers (15 base + 6 features Climat).

## Wiring 11-node (complet)
- `extended_v6_wind=True` → builder ajoute U850, V850
- 2 metapaths injectés (U850_spat, V850_spat) → num_vars 9 → **11**
- routing canaux : U850 ← u_850/500/250, V850 ← v_850/500/250
- **G_phys 11×11** : `physics_prior.VAR_LABELS_V6` / `EXPECTED_EDGES_V6` (14 arêtes : 10 humide-QG + 4 vent U850/V850→IVT/SP_HR)

## Garde-fous pré-enregistrés (`V6_PRIME_seuils_preregistered.json`)
- **M1** normalisation LR z-score figée train ; **M2** concat centralisé ; **M3** parité inférence (sample lève si lr_fields=None)
- **A1** (mu_HR→0) / **A2** (LR→0, post-norm) monitorées **dès le SMOKE** (M6)
- Métrique **co-primaire** : per-gridpoint ETCCDI (battre 0.816, ne pas régresser < V5=0.841) + pooled (0.550)
- Ablations A1/A2 câblées end-to-end (Cell 11)


In [ ]:
# >>> Cell 1 : Bootstrap Colab + git sync (P0 fix : sys.path src/)
import os, sys, subprocess
from pathlib import Path

GIT_URL = "https://github.com/leonelkenfack/stcdgm.git"
GIT_BRANCH = "four-node-causal"
REPO_DIR = "/content/climate_data"
DRIVE_ROOT = "/content/drive/MyDrive/climate_data"

if not Path("/content/drive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not Path(REPO_DIR).exists():
    subprocess.check_call(["git", "clone", "--depth=200", "-b", GIT_BRANCH, GIT_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin"])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", GIT_BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "origin", GIT_BRANCH])

# P0 FIX (audit IA) : add BOTH repo root (path_c_plus, scripts) AND repo/src (st_cdgm)
for _p in (REPO_DIR, str(Path(REPO_DIR) / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# P0 FIX : chdir vers la racine du repo — sinon les chemins relatifs
# (config/*.yaml, données, checkpoints) échouent depuis /content (Colab CWD).
os.chdir(REPO_DIR)

# P0 FIX (revue dev/ML 2026-07) : liste de deps EPINGLEE identique au 9-node qui
# tournait. La liste a 4 items causait la cascade "une erreur par run" :
#   cftime  -> decode calendrier 'noleap' (sinon crash sur TOUT open NetCDF)
#   xbatcher-> NetCDFDataPipeline.__init__ leve ImportError sans lui
#   diffusers==0.36.0 + stack epingle -> build UNet2DConditionModel stable
#   h5netcdf/netcdf4 -> moteurs de lecture + fallback robuste du pipeline
try:
    import torch_geometric, cftime, h5netcdf, xbatcher, diffusers, omegaconf  # noqa: F401
    print("[Cell 1] deps critiques OK — pip install saute.")
except Exception as _e:
    print(f"[Cell 1] pip install requis : {_e}")
    _EXTRA = ["omegaconf==2.3.0", "hydra-core==1.3.2", "diffusers==0.36.0",
              "transformers==4.57.6", "accelerate==1.12.0", "huggingface-hub==0.36.0",
              "safetensors==0.7.0", "xbatcher", "webdataset", "cftime", "h5netcdf",
              "netcdf4", "numcodecs", "scipy", "torch-geometric", "xformers"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "--no-warn-script-location", *_EXTRA])

_sha = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"]).decode().strip()
print(f"[Cell 1] Bootstrap OK — commit {_sha[:8]} — sys.path has src/ (P0 fix)")


In [ ]:
# >>> Cell 2 : Config + V6' constants (11-node Stage 1, 22-ch Stage 2 cond)
import torch, numpy as np
from omegaconf import OmegaConf

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE_MODE = False   # True → few epochs/batches for the mandatory smoke

from st_cdgm.v6_constants import (
    NONCAUSAL_15_VARS, V6_LR_VARS_OBLIGATORY, V6_LR_VARS_FULL,
    V6_NODE_CHANNEL_ROUTING, V6_NUM_ENCODER_VARS, assert_lr_vars_match,
)

USE_IVT_72H_BONUS = True
# Stage 1 drivers = full augmented LR (21 obligatory, + IVT-72h bonus if enabled)
STAGE1_LR_VARS = list(V6_LR_VARS_FULL) if USE_IVT_72H_BONUS else list(V6_LR_VARS_OBLIGATORY)
# Stage 2 conditioning = same set (full LR)
STAGE2_COND_LR_VARS = list(STAGE1_LR_VARS)
LR_COND_CHANNELS = len(STAGE2_COND_LR_VARS)
print(f"[Cell 2] Stage 1 LR vars = {len(STAGE1_LR_VARS)} (drivers)  |  "
      f"Stage 2 LR conditioning channels = {LR_COND_CHANNELS}")

# --- OOD parameterization (fix ML : évite le re-run manuel skippable) -------
# Pour l'OOD, changer UNIQUEMENT GCM_ID — les stats de normalisation restent
# celles d'ACCESS-CM2 (fichiers explicites, cf. Cell 3). PAS de re-norm per-GCM.
GCM_ID = "ACCESS-CM2"           # "EC-Earth3" pour le run OOD
IS_OOD_RUN = GCM_ID != "ACCESS-CM2"

CONFIG = OmegaConf.load("config/training_config.yaml")
CONFIG = OmegaConf.merge(CONFIG, OmegaConf.load("config/training_config_corrdiff_normal.yaml"))
OmegaConf.set_struct(CONFIG, False)
CONFIG.data.lr_variables = STAGE1_LR_VARS   # 11-node Stage 1 sees the full LR

EXTENDED_9NODE = True
EXTENDED_V6_WIND = True   # <<< FULL V6 : U850/V850 as graph nodes

SEEDS = [42]
K9_DATES = {"train": ("1980-01-01","2009-12-31"), "val": ("2010-01-01","2011-12-31"),
            "test": ("2012-01-01","2013-12-31"), "holdout": ("2014-01-01","2014-12-31")}
print("[Cell 2] V6' FULL config ready (11-node + full-LR conditioning)")


In [ ]:
# >>> Cell 2B : Data provisioning — download base si absent + preproc v6 si absent
# Rend le notebook end-to-end : garantit que lr_{GCM}_v6.nc, static_HR_v6.nc et le
# HR de base existent AVANT Cell 3. Reutilise le mecanisme Zenodo du 9-node.
import os, sys, glob, time, subprocess
import urllib.request as _ureq, urllib.error as _uerr
from pathlib import Path

DATA_ROOT = Path(f"{DRIVE_ROOT}/data")
for _sub in ("train", "static_predictors", "test"):
    (DATA_ROOT / _sub).mkdir(parents=True, exist_ok=True)

# --- Streaming download avec reprise + backoff (porte du 9-node Cell 3) ------
def _stream_dl(url, dest, retries=5, chunk=1024*1024):
    dest = Path(dest); dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(1, retries + 1):
        already = part.stat().st_size if part.exists() else 0
        req = _ureq.Request(url)
        if already > 0:
            req.add_header("Range", f"bytes={already}-")
        try:
            with _ureq.urlopen(req, timeout=30) as resp:
                mode = "ab" if already > 0 else "wb"
                with open(part, mode) as f:
                    got = already; last = time.time(); last_b = got
                    while True:
                        c = resp.read(chunk)
                        if not c: break
                        f.write(c); got += len(c)
                        if time.time() - last >= 5:
                            sp = (got - last_b) / (time.time() - last) / 1e6
                            print(f"    {got/1e6:7.1f} MB -- {sp:5.1f} MB/s")
                            last = time.time(); last_b = got
            os.replace(part, dest)
            print(f"  OK {dest.name} ({dest.stat().st_size/1e6:.1f} MB)")
            return True
        except (_uerr.HTTPError, _uerr.URLError, TimeoutError, ConnectionError) as e:
            wait = min(60, 2 ** attempt)
            print(f"  WARN {type(e).__name__}: {e} -- retry {wait}s"); time.sleep(wait)
    raise RuntimeError(f"Echec download {url}")

# --- Recherche robuste d'un fichier de base (glob recursif) ------------------
def _find(patterns, roots):
    for r in roots:
        for pat in patterns:
            hits = sorted(glob.glob(str(Path(r) / "**" / pat), recursive=True))
            if hits:
                return hits[0]
    return None

_ROOTS = [DATA_ROOT, DRIVE_ROOT]

# ACCESS-CM2 (in-distribution) : base LR/HR telechargeables depuis Zenodo.
# Autres GCM (OOD) : doivent deja etre sur le Drive (pas sur Zenodo).
_ZENODO = {
    "predictor_ACCESS-CM2_hist.nc":
        "https://zenodo.org/records/10889046/files/predictor_ACCESS-CM2_hist.nc?download=1",
    "pr_ACCESS-CM2_hist.nc":
        "https://zenodo.org/records/10889046/files/pr_ACCESS-CM2_hist.nc?download=1",
}

if GCM_ID == "ACCESS-CM2":
    _lr_names = ["predictor_ACCESS-CM2_hist.nc", "lr_ACCESS-CM2.nc"]
    _hr_names = ["pr_ACCESS-CM2_hist.nc", "hr_NIWA-REMS.nc"]
else:
    # OOD : conventions test/ du 9-node (compressed) + variantes plausibles
    _lr_names = [f"{GCM_ID}_histupdated_compressed.nc", f"predictor_{GCM_ID}*.nc", f"lr_{GCM_ID}.nc"]
    _hr_names = [f"{GCM_ID}_historical_precip_compressed.nc", f"pr_{GCM_ID}*.nc", f"hr_{GCM_ID}.nc"]

BASE_LR     = _find(_lr_names, _ROOTS)
BASE_HR     = _find(_hr_names, _ROOTS)
BASE_STATIC = _find(["*Invariant*.nc", "*invariant*.nc", "static_HR.nc",
                     "ERA5_eval_ccam_12km*.nc"], _ROOTS)

# --- Download Zenodo si absent (ACCESS-CM2 uniquement) -----------------------
if BASE_LR is None and GCM_ID == "ACCESS-CM2":
    BASE_LR = str(DATA_ROOT / "train" / "predictor_ACCESS-CM2_hist.nc")
    print(f"[Cell 2B] base LR absent -> Zenodo -> {BASE_LR}")
    _stream_dl(_ZENODO["predictor_ACCESS-CM2_hist.nc"], BASE_LR)
if BASE_HR is None and GCM_ID == "ACCESS-CM2":
    BASE_HR = str(DATA_ROOT / "train" / "pr_ACCESS-CM2_hist.nc")
    print(f"[Cell 2B] base HR absent -> Zenodo -> {BASE_HR}")
    _stream_dl(_ZENODO["pr_ACCESS-CM2_hist.nc"], BASE_HR)

# --- Diagnostics durs (fail loud, jamais de fallback silencieux) -------------
if BASE_LR is None:
    raise FileNotFoundError(
        f"[Cell 2B] LR de base introuvable pour {GCM_ID}. "
        f"Cherche {_lr_names} sous {[str(r) for r in _ROOTS]}. "
        f"OOD : deposer le fichier LR du GCM sur le Drive (non Zenodo).")
if BASE_HR is None:
    raise FileNotFoundError(
        f"[Cell 2B] HR de base introuvable pour {GCM_ID}. Cherche {_hr_names}.")
if BASE_STATIC is None:
    raise FileNotFoundError(
        "[Cell 2B] static HR (orographie/masque) introuvable. "
        "Cherche *Invariant*.nc — doit venir du run noncausal (non Zenodo).")
print(f"[Cell 2B] BASE_LR     = {BASE_LR}")
print(f"[Cell 2B] BASE_HR     = {BASE_HR}")
print(f"[Cell 2B] BASE_STATIC = {BASE_STATIC}")

# --- Sorties v6 : preproc si absent -----------------------------------------
LR_PATH_V6  = f"{DRIVE_ROOT}/lr_{GCM_ID}_v6.nc"
STATIC_PATH = f"{DRIVE_ROOT}/static_HR_v6.nc"
HR_PATH     = str(BASE_HR)   # HR de base reel (remplace l'ancien hr_NIWA-REMS.nc invente)

# --- Auto-detection des noms de variables (insensible casse : t_850 vs T_850) --
import xarray as _xr2
def _resolve(ds, cands):
    _low = {v.lower(): v for v in ds.data_vars}
    for c in cands:
        if c in ds.data_vars: return c
        if c.lower() in _low: return _low[c.lower()]
    return None

# decode_times=False : on ne lit que des noms de variables ici — pas besoin de
# decoder le calendrier 'noleap' (qui exigerait cftime). Evite un crash inutile.
_lr_ds = _xr2.open_dataset(str(BASE_LR), decode_times=False)
_st_ds = _xr2.open_dataset(str(BASE_STATIC), decode_times=False)
_VNAMES = {
    "--w-850-name": _resolve(_lr_ds, ["w_850", "wap_850"]),
    "--w-500-name": _resolve(_lr_ds, ["w_500", "wap_500"]),
    "--t-850-name": _resolve(_lr_ds, ["t_850", "T_850", "ta_850"]),
    "--t-500-name": _resolve(_lr_ds, ["t_500", "T_500", "ta_500"]),
    "--q-850-name": _resolve(_lr_ds, ["q_850", "hus_850"]),
    "--q-500-name": _resolve(_lr_ds, ["q_500", "hus_500"]),
    "--u-850-name": _resolve(_lr_ds, ["u_850", "ua_850"]),
    "--v-850-name": _resolve(_lr_ds, ["v_850", "va_850"]),
    "--orog-name":  _resolve(_st_ds, ["orog", "he", "topo", "elevation", "z"]),
    "--land-sea-mask-name": _resolve(_st_ds, ["sftlf", "land_sea_mask", "lsm", "mask"]) or "sftlf",
}
print(f"[Cell 2B] LR vars dispo : {list(_lr_ds.data_vars)}")
print(f"[Cell 2B] static vars dispo : {list(_st_ds.data_vars)}")
_missing = [k for k, v in _VNAMES.items() if v is None and k != "--land-sea-mask-name"]
if _missing:
    _lr_ds.close(); _st_ds.close()
    raise RuntimeError(f"[Cell 2B] variables preproc introuvables : {_missing} — "
                       f"ajuster les candidats de _resolve dans Cell 2B.")
_NAME_ARGS = []
for _k, _v in _VNAMES.items():
    if _v is not None: _NAME_ARGS += [_k, _v]
print(f"[Cell 2B] mapping noms : { {k: v for k, v in _VNAMES.items()} }")

# --- P1 fix (revue dev) : le preproc HARD-crash si l'orographie n'a pas des
# dims litteralement 'lat'/'lon' (gradient orographique signe). Rename defensif
# vers un fichier static temporaire si les dims sont y/x ou rlat/rlon.
_orog_name = _VNAMES["--orog-name"]
_st_dims = list(_st_ds[_orog_name].dims)
_RENAME = {}
for _cand, _canon in [("y","lat"),("x","lon"),("rlat","lat"),("rlon","lon"),
                      ("latitude","lat"),("longitude","lon")]:
    if _cand in _st_dims and _canon not in _st_dims:
        _RENAME[_cand] = _canon
if _RENAME:
    _static_fixed = f"{DRIVE_ROOT}/static_HR_dimfix.nc"
    print(f"[Cell 2B] rename dims static {_RENAME} -> {_static_fixed}")
    _xr2.open_dataset(str(BASE_STATIC), decode_times=False).rename(_RENAME).to_netcdf(_static_fixed)
    BASE_STATIC = _static_fixed
_lr_ds.close(); _st_ds.close()

# --- P2 fix (revue ML) : ne pas reutiliser un cache v6 PERIME (ex. 21 vars,
# genere sans le bonus IVT-72h) — valider qu'il contient TOUTES les STAGE1_LR_VARS.
def _v6_has_all_vars(path):
    if not Path(path).exists(): return False
    _d = _xr2.open_dataset(str(path), decode_times=False)
    _ok = all(v in _d.data_vars for v in STAGE1_LR_VARS)
    _d.close()
    return _ok

_need_preproc = not (_v6_has_all_vars(LR_PATH_V6) and Path(STATIC_PATH).exists())
if _need_preproc and Path(LR_PATH_V6).exists():
    print(f"[Cell 2B] cache v6 present mais incomplet (vars manquantes) -> re-preproc")
if _need_preproc:
    print(f"[Cell 2B] preproc v6 ({GCM_ID}) -> {LR_PATH_V6}")
    _r = subprocess.run(
        [sys.executable, "path_c_plus/scripts/preprocess_v6_lr.py",
         "--lr-path",          str(BASE_LR),
         "--static-hr-path",   str(BASE_STATIC),
         "--out-lr-augmented", LR_PATH_V6,
         "--out-static-hr-v6", STATIC_PATH,
         *_NAME_ARGS],
        capture_output=True, text=True,
    )
    if _r.stdout: print(_r.stdout[-3000:])
    if _r.returncode != 0:
        print("----- STDERR preproc -----")
        print(_r.stderr[-5000:])
        raise RuntimeError("[Cell 2B] preproc v6 a echoue — voir STDERR ci-dessus")
    print("[Cell 2B] preproc v6 termine")
else:
    print(f"[Cell 2B] sorties v6 deja presentes : {LR_PATH_V6}")

assert Path(LR_PATH_V6).exists(), f"preproc n'a pas produit {LR_PATH_V6}"
assert Path(STATIC_PATH).exists(), f"preproc n'a pas produit {STATIC_PATH}"
# Validation dure : le fichier v6 doit exposer les 22 STAGE1_LR_VARS (sinon
# KeyError plus tard dans Cell 3 apres avoir gaspille le preproc). Fail loud ici.
_dchk = _xr2.open_dataset(str(LR_PATH_V6), decode_times=False)
_vmiss = [v for v in STAGE1_LR_VARS if v not in _dchk.data_vars]
_dvars = list(_dchk.data_vars); _dchk.close()
if _vmiss:
    raise RuntimeError(
        f"[Cell 2B] {LR_PATH_V6} n'expose pas {_vmiss}. Presentes : {_dvars}. "
        f"Cause probable : casse des noms de base (ex. T_850 vs t_850) preservee "
        f"par le preproc (ds_lr.copy). Etendre le renommage canonique.")
print(f"[Cell 2B] provisioning OK — {len(STAGE1_LR_VARS)} vars presentes — Cell 3 peut consommer")


In [ ]:
# >>> Cell 3 : Pipeline + 11-node builder + metapath injection + routing
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from omegaconf import OmegaConf as _OC

# Chemins fournis par Cell 2B (provisioning : download base + preproc v6).
# Fallback defensif si Cell 3 est lancee seule, mais Cell 2B doit tourner avant.
LR_PATH_V6  = globals().get("LR_PATH_V6",  f"{DRIVE_ROOT}/lr_{GCM_ID}_v6.nc")
HR_PATH     = globals().get("HR_PATH",      f"{DRIVE_ROOT}/data/train/pr_{GCM_ID}_hist.nc")
STATIC_PATH = globals().get("STATIC_PATH",  f"{DRIVE_ROOT}/static_HR_v6.nc")
assert Path(LR_PATH_V6).exists(), f"{LR_PATH_V6} absent — lancer Cell 2B (provisioning) d'abord"
# --- P1-C fix (audit IA) + fix OOD (audit ML) --------------------------------
# Les anciens means/stds (15 vars) ne couvrent PAS les 22 vars V6' → KeyError.
# Protocole : on génère UNE FOIS des stats v6 explicites sur la fenêtre train
# d'ACCESS-CM2, sauvegardées sur Drive, et TOUT run (train ET OOD EC-Earth3)
# les charge explicitement. Jamais de fallback silencieux None (qui ferait
# recalculer les stats sur le GCM OOD = fuite de re-normalisation per-GCM).
import xarray as _xr
MEANS_PATH_V6 = f"{DRIVE_ROOT}/train/means_ACCESS-CM2_v6.nc"
STDS_PATH_V6  = f"{DRIVE_ROOT}/train/stds_ACCESS-CM2_v6.nc"

if not (Path(MEANS_PATH_V6).exists() and Path(STDS_PATH_V6).exists()):
    if IS_OOD_RUN:
        raise FileNotFoundError(
            f"OOD run ({GCM_ID}) : les stats train ACCESS-CM2 v6 sont OBLIGATOIRES "
            f"({MEANS_PATH_V6}). Lancer d'abord le run in-distribution qui les génère. "
            f"Recalculer les stats sur {GCM_ID} masquerait les biais moyens (audit ML)."
        )
    print("[Cell 3] Génération des stats train v6 (une fois) ...")
    os.makedirs(os.path.dirname(MEANS_PATH_V6), exist_ok=True)   # fix revue dev : Drive propre
    _ds_access = _xr.open_dataset(f"{DRIVE_ROOT}/lr_ACCESS-CM2_v6.nc")
    _tr = _ds_access.sel(time=slice(K9_DATES["train"][0], K9_DATES["train"][1]))
    _tr = _tr[STAGE1_LR_VARS]
    _tr.mean(dim="time").to_netcdf(MEANS_PATH_V6)
    _tr.std(dim="time").to_netcdf(STDS_PATH_V6)
    _ds_access.close()
    print(f"[Cell 3] stats v6 sauvegardées : {MEANS_PATH_V6}")
# Vérifie que les stats couvrent bien les 22 vars (fail loud, pas de KeyError tardif)
_m_check = _xr.open_dataset(MEANS_PATH_V6)
_missing_stats = [v for v in STAGE1_LR_VARS if v not in _m_check.data_vars]
_m_check.close()
assert not _missing_stats, f"Stats v6 incomplètes, vars manquantes : {_missing_stats}"

pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH_V6, hr_path=HR_PATH,
    static_path=STATIC_PATH if Path(STATIC_PATH).exists() else None,
    seq_len=int(CONFIG.data.seq_len),
    baseline_strategy=str(CONFIG.data.baseline_strategy),
    baseline_factor=int(CONFIG.data.baseline_factor),
    normalize=bool(CONFIG.data.normalize),
    nan_fill_strategy=str(CONFIG.data.nan_fill_strategy),
    precipitation_delta=float(CONFIG.data.precipitation_delta),
    lr_variables=STAGE1_LR_VARS,
    hr_variables=list(CONFIG.data.hr_variables),
    static_variables=list(CONFIG.data.static_variables) if CONFIG.data.get("static_variables") else [],
    means_path=MEANS_PATH_V6,   # EXPLICITE — jamais None (M1 + fix OOD)
    stds_path=STDS_PATH_V6,
    train_start_date=K9_DATES["train"][0], train_end_date=K9_DATES["train"][1],
    val_start_date=K9_DATES["val"][0],     val_end_date=K9_DATES["val"][1],
    test_start_date=K9_DATES["test"][0],   test_end_date=K9_DATES["test"][1],
    temporal_holdout_start_date=K9_DATES["holdout"][0],
    temporal_holdout_end_date=K9_DATES["holdout"][1],
)

# --- 11-node builder : 9-node + U850/V850 wind nodes -----------------------
builder = HeteroGraphBuilder(
    lr_shape=tuple(CONFIG.graph.lr_shape), hr_shape=tuple(CONFIG.graph.hr_shape),
    static_dataset=pipeline.get_static_dataset(), include_mid_layer=True,
    extended_9node=True, extended_v6_wind=True,
)
print(f"[Cell 3] builder dynamic nodes = {builder.dynamic_node_types}")
assert "U850" in builder.dynamic_node_types and "V850" in builder.dynamic_node_types

# --- Inject humid (9-node) + wind (V6) spatial metapaths -------------------
_existing = {m.name for m in CONFIG.encoder.metapaths}
_new_mps = [
    {"name": "Q850_spat", "src": "Q850", "relation": "spat_adj", "target": "Q850", "pool": "mean"},
    {"name": "W500_spat", "src": "W500", "relation": "spat_adj", "target": "W500", "pool": "mean"},
    {"name": "IVT_spat",  "src": "IVT",  "relation": "spat_adj", "target": "IVT",  "pool": "mean"},
    {"name": "U850_spat", "src": "U850", "relation": "spat_adj", "target": "U850", "pool": "mean"},
    {"name": "V850_spat", "src": "V850", "relation": "spat_adj", "target": "V850", "pool": "mean"},
]
for _m in _new_mps:
    if _m["name"] not in _existing:
        CONFIG.encoder.metapaths.append(_OC.create(_m))
print(f"[Cell 3] metapaths -> {[m.name for m in CONFIG.encoder.metapaths]}")

# --- Per-node channel routing (V6) -----------------------------------------
_LR_VARS = list(CONFIG.data.lr_variables)
_VI = {v: i for i, v in enumerate(_LR_VARS)}
def _idx(names): return [_VI[v] for v in names if v in _VI]
_ROUTE_IDX = {node: _idx(chans) for node, chans in V6_NODE_CHANNEL_ROUTING.items()}
_IVT_LEVELS = [lev for lev in ("850","500","250")
               if f"q_{lev}" in _VI and f"u_{lev}" in _VI and f"v_{lev}" in _VI]
print(f"[Cell 3] routing idx = {_ROUTE_IDX} | IVT levels = {_IVT_LEVELS}")

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, _VI[f"q_{lev}"]]; u = lr0[:, _VI[f"u_{lev}"]]; v = lr0[:, _VI[f"v_{lev}"]]
        term = q * torch.sqrt(u*u + v*v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None: acc = torch.zeros(lr0.shape[0], device=lr0.device, dtype=lr0.dtype)
    acc = (acc - acc.mean()) / (acc.std() + 1e-6)
    return acc.unsqueeze(1)

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample["lr"]                       # [seq, C_LR, lat, lon]
    seq_len = lr_seq.shape[0]
    lr_nodes = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes, dim=0)    # RCN drivers (full LR)
    lr0 = lr_nodes[0]
    _ivt = _compute_ivt_nodes(lr0)
    dyn = {}
    for nt in builder.dynamic_node_types:
        if nt in _ROUTE_IDX and _ROUTE_IDX[nt]:
            dyn[nt] = lr0[:, _ROUTE_IDX[nt]]     # Q850/W500/U850/V850 <- routed channels
        elif nt == "IVT":
            dyn[nt] = _ivt
        else:
            dyn[nt] = lr0                        # GP850/GP500/GP250 <- full LR
    hetero = builder.prepare_step_data(dyn).to(device)
    return {
        "lr": lr_tensor, "residual": sample["residual"],
        "baseline": sample.get("baseline"), "hetero": hetero,
        "time": sample.get("time"),
        "lr_grid": lr_seq,   # V6' : raw LR conditioning grid [seq, C_LR, H_LR, W_LR]
    }

def iterate_batches_v6(ds, builder, device):
    for s in ds:
        yield [convert_sample_to_batch(s, builder, device)]

train_dataset = pipeline.build_sequence_dataset(split="train", seq_len=int(CONFIG.data.seq_len), stride=int(CONFIG.data.stride), as_torch=True)
val_dataset   = pipeline.build_sequence_dataset(split="val",   seq_len=int(CONFIG.data.seq_len), stride=int(CONFIG.data.stride), as_torch=True)
test_dataset  = pipeline.build_sequence_dataset(split="test",  seq_len=int(CONFIG.data.seq_len), stride=int(CONFIG.data.stride), as_torch=True)
print("[Cell 3] datasets ready (11-node, lr_grid conditioning attached)")


In [ ]:
# >>> Cell 4 : Stack (encoder 11 vars + RCN 11x11 + regression_head + diffusion V6')
from st_cdgm.models.intelligible_encoder import IntelligibleVariableEncoder, IntelligibleVariableConfig
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig

RCN_DRIVER_DIM = None

def build_fresh_stack(seed: int):
    global RCN_DRIVER_DIM
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    allowed = set(builder.dynamic_node_types) | set(builder.static_node_types)
    enc_cfgs = [IntelligibleVariableConfig(name=m.name, meta_path=(m.src, m.relation, m.target),
                                            pool=m.get("pool", "mean"))
                for m in CONFIG.encoder.metapaths if m.src in allowed and m.target in allowed]
    # static SP_HR conditioning variable (as in 9-node _build_encoder)
    if pipeline.get_static_dataset() is not None:
        enc_cfgs.append(IntelligibleVariableConfig(name="static",
            meta_path=("SP_HR", "causes", "GP850"), pool="mean"))
    encoder = IntelligibleVariableEncoder(configs=enc_cfgs,
        hidden_dim=int(CONFIG.encoder.hidden_dim),
        conditioning_dim=int(CONFIG.encoder.conditioning_dim)).to(DEVICE)
    num_vars = len(enc_cfgs)
    assert num_vars == V6_NUM_ENCODER_VARS, f"num_vars={num_vars} != {V6_NUM_ENCODER_VARS} (11)"
    print(f"   encoder : {num_vars} intelligible variables (11-node V6)")

    # P0-A fix (audit IA) : lr_grid_to_nodes exige un tenseur 3D [C, H, W] —
    # un zeros 2D (lr_shape seul) levait ValueError. Probe avec les 22 canaux.
    _probe = builder.lr_grid_to_nodes(
        torch.zeros(len(STAGE1_LR_VARS), *tuple(CONFIG.graph.lr_shape))
    )
    RCN_DRIVER_DIM = _probe.shape[-1]
    rcn_cell = RCNCell(num_vars=num_vars, hidden_dim=int(CONFIG.rcn.hidden_dim),
        driver_dim=RCN_DRIVER_DIM, reconstruction_dim=RCN_DRIVER_DIM,
        dropout=float(CONFIG.rcn.dropout)).to(DEVICE)
    rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get("detach_interval"))
    print(f"   rcn_cell A_dag shape = {tuple(rcn_cell.A_dag.shape)} (target 11x11)")

    rh = CONFIG.two_stage.regression_head
    regression_head = GraphToGridDecoder(d_model=int(rh.d_model),
        hr_h=int(CONFIG.diffusion.height), hr_w=int(CONFIG.diffusion.width),
        intermediate_h=int(rh.intermediate_h), intermediate_w=int(rh.intermediate_w),
        n_heads=int(rh.n_heads), refine_channels=int(rh.refine_channels),
        output_channels=1).to(DEVICE)

    UNET = OmegaConf.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
    for k in ("down_block_types", "up_block_types"):
        if k in UNET and isinstance(UNET[k], list): UNET[k] = tuple(UNET[k])
    UNET["projection_class_embeddings_input_dim"] = num_vars * int(CONFIG.diffusion.conditioning_dim)
    diffusion = CausalDiffusionDecoder(
        in_channels=1, conditioning_dim=int(CONFIG.diffusion.conditioning_dim),
        height=int(CONFIG.diffusion.height), width=int(CONFIG.diffusion.width),
        unet_kwargs=UNET, scheduler_type="edm_karras",
        edm_config=EDMConfig.from_yaml_dict(CONFIG.diffusion.get("edm", {})),
        causal_concat=True,
        lr_conditioning_channels=LR_COND_CHANNELS,   # <<< V6' PIVOT
        # fix mémoire GPU : REQUIS pour batch=64 sur A100 (yaml corrdiff_normal).
        # Recompute activations au backward -> ~30% plus lent, ~50% moins de VRAM.
        use_gradient_checkpointing=bool(CONFIG.diffusion.get("use_gradient_checkpointing", True)),
    ).to(DEVICE)
    print(f"   diffusion conv_in in_channels = {diffusion.unet.conv_in.in_channels} (=3+{LR_COND_CHANNELS})")
    return dict(encoder=encoder, rcn_cell=rcn_cell, rcn_runner=rcn_runner,
                regression_head=regression_head, diffusion=diffusion, num_vars=num_vars)

print("[Cell 4] build_fresh_stack (11-node + lr_conditioning) defined")


In [ ]:
# >>> Cell 5 : Helpers (REAL imports) + G_phys 11x11 + schedule
import copy, subprocess as _sp
from st_cdgm.training.training_loop import train_epoch_stage1
from st_cdgm.training.two_stage import (
    freeze_stage1, causal_ablation_check, precompute_stage1_outputs, train_epoch_stage2_cached,
    calibrate_sigma_data_two_stage,   # safeguard 9-node reintegre (revue ML)
)
from st_cdgm.training.physics_prior import build_physical_mask, VAR_LABELS_V6, EXPECTED_EDGES_V6
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES
from scripts.finetune_stage1_bundle_b import schedule_lambdas, DEFAULT_HYPERPARAMS

from st_cdgm.v6_constants import V6_LAMBDA_L1_START, V6_LAMBDA_L1_END

HP = copy.deepcopy(DEFAULT_HYPERPARAMS)
HP.update({
    "lambda_dag_prior": PATHCPLUS_HYPERPARAM_OVERRIDES["lambda_dag_prior"],  # 0.40
    # P2-i fix (audit IA) : le plan V6' §2 spécifie λ_l1 ×0.7 (adaptation
    # 11-node) — utiliser les constantes V6, pas les overrides 9-node.
    # (Audit Math : strictement inutile car gradient L1 par-entrée, mais
    # inoffensif — on suit le plan pré-enregistré.)
    "lambda_l1_start":  V6_LAMBDA_L1_START,   # 0.028 (9-node : 0.04)
    "lambda_l1_end":    V6_LAMBDA_L1_END,     # 0.0035 (9-node : 0.005)
    "g_phys_alpha":     PATHCPLUS_HYPERPARAM_OVERRIDES["g_phys_alpha"],
    "dag_gate_warmup_start_epoch": PATHCPLUS_HYPERPARAM_OVERRIDES["dag_gate_warmup_start_epoch"],
    "dag_gate_warmup_end_epoch":   PATHCPLUS_HYPERPARAM_OVERRIDES["dag_gate_warmup_end_epoch"],
})

# G_phys 11x11 (V6 : 9-node humid-QG + 4 wind edges U850/V850 -> IVT/SP_HR)
G_phys = build_physical_mask(num_vars=11, var_labels=VAR_LABELS_V6,
                             expected_edges=EXPECTED_EDGES_V6).to(DEVICE)
print(f"[Cell 5] G_phys 11x11 : |edges|={int((G_phys != 0).sum())} (expect 14)")
assert G_phys.shape == (11, 11)

PRE_REG_COMMIT = _sp.check_output(["git","-C",REPO_DIR,"rev-parse","--short","HEAD"]).decode().strip()
print(f"[Cell 5] pre-registration commit = {PRE_REG_COMMIT}")
print("[Cell 5] thresholds : path_c_plus/audit/V6_PRIME_seuils_preregistered.json")


In [ ]:
# >>> Cell 6 : Stage 1 (11-node) training from scratch (seed 42)
import torch.nn.functional as F, os
SEED = 42
S1_EPOCHS = 3 if SMOKE_MODE else 30

# --- Checkpointing (safeguard 9-node reintegre — resiste aux deconnexions Colab)
CKPT_DIR = f"{DRIVE_ROOT}/oracle_v6_prime/seed_42"; os.makedirs(CKPT_DIR, exist_ok=True)
S1_CKPT  = f"{CKPT_DIR}/stage1_seed42.pth"           # final (poids figes + sigma)
S1_PROG  = f"{CKPT_DIR}/stage1_progress_seed42.pth"  # progression par epoque

def _atomic_save(obj, path):
    # ecriture .tmp + os.replace : jamais de checkpoint corrompu si Colab meurt
    # en pleine ecriture (le fichier final reste l'ancien complet).
    _tmp = path + ".tmp"
    torch.save(obj, _tmp); os.replace(_tmp, path)

stack = build_fresh_stack(SEED)
encoder, rcn_cell = stack["encoder"], stack["rcn_cell"]
rcn_runner, regression_head = stack["rcn_runner"], stack["regression_head"]
diffusion = stack["diffusion"]

ts = CONFIG.two_stage.stage1
# fix (grep-all-callsites) : CONFIG.training.learning_rate n'existe pas dans le
# merge — le 9-node utilise two_stage.stage1.lr (0.0003) + stage1.weight_decay.
opt_s1 = torch.optim.AdamW(
    list(encoder.parameters()) + list(rcn_cell.parameters()) + list(regression_head.parameters()),
    lr=float(ts.lr), weight_decay=float(ts.get("weight_decay", 1e-4)))

def _save_stage1_progress(epoch_done):
    _atomic_save({"encoder_state_dict": encoder.state_dict(),
                  "rcn_cell_state_dict": rcn_cell.state_dict(),
                  "regression_head_state_dict": regression_head.state_dict(),
                  "opt_state_dict": opt_s1.state_dict(), "epoch_done": epoch_done}, S1_PROG)

# RESUME (3 cas) : (a) Stage 1 fini -> S1_CKPT existe -> skip. (b) partiel ->
# S1_PROG existe -> reprend a l'epoque exacte. (c) rien -> from scratch.
S1_RESUMED = (not SMOKE_MODE) and os.path.exists(S1_CKPT)
_s1_start = 0
if S1_RESUMED:
    _s1 = torch.load(S1_CKPT, map_location=DEVICE, weights_only=False)
    encoder.load_state_dict(_s1["encoder_state_dict"])
    rcn_cell.load_state_dict(_s1["rcn_cell_state_dict"])
    regression_head.load_state_dict(_s1["regression_head_state_dict"])
    S1_SIGMA_DATA = float(_s1.get("sigma_data", float("nan")))
    print(f"[Cell 6] Stage 1 DEJA FINI ({S1_CKPT}) — entrainement saute "
          f"(sigma_data={S1_SIGMA_DATA:.5f})")
elif (not SMOKE_MODE) and os.path.exists(S1_PROG):
    _sp = torch.load(S1_PROG, map_location=DEVICE, weights_only=False)
    encoder.load_state_dict(_sp["encoder_state_dict"])
    rcn_cell.load_state_dict(_sp["rcn_cell_state_dict"])
    regression_head.load_state_dict(_sp["regression_head_state_dict"])
    try: opt_s1.load_state_dict(_sp["opt_state_dict"])
    except Exception as _e: print(f"[Cell 6] opt_s1 state non repris ({_e})")
    _s1_start = int(_sp.get("epoch_done", 0))
    print(f"[Cell 6] Stage 1 RESUME a l'epoque {_s1_start}/{S1_EPOCHS}")

for ep in ([] if S1_RESUMED else range(_s1_start, S1_EPOCHS)):
    sch = schedule_lambdas(ep, S1_EPOCHS, HP)
    m = train_epoch_stage1(
        encoder=encoder, rcn_runner=rcn_runner, regression_head=regression_head,
        optimizer=opt_s1, data_loader=iterate_batches_v6(train_dataset, builder, DEVICE),
        device=DEVICE, epoch_idx=ep,
        lambda_reg=float(ts.lambda_reg), beta_rec=float(ts.beta_rec),
        gamma_dag_max=float(ts.gamma_dag_max), gamma_dag_warmup_epochs=int(ts.gamma_dag_warmup_epochs),
        lambda_l1=float(sch["lambda_l1"]), lambda_dag_prior=float(sch["lambda_dag_prior"]),
        dag_prior=G_phys, dag_grad_gate_value=float(sch["dag_grad_gate"]),
        abort_on_collapse=True, collapse_threshold=0.05,
        dag_floor_projection=True, dag_floor_min_norm=0.10,
        gradient_clipping=CONFIG.training.gradient_clipping)
    if not SMOKE_MODE:
        _save_stage1_progress(ep + 1)   # checkpoint APRES CHAQUE epoque (choix user)
    if (ep+1) % 5 == 0 or ep == 0:
        # fix (grep-all-callsites) : train_epoch_stage1 retourne 'loss'/'loss_rec'/
        # 'loss_dag' + sante DAG (a_*_end), PAS 'loss_total'.
        print(f"  S1 ep{ep+1}/{S1_EPOCHS} loss={m.get('loss', float('nan')):.4f} "
              f"rec={m.get('loss_rec', float('nan')):.4f} dag={m.get('loss_dag', float('nan')):+.4f} "
              f"| ||A||_F={m.get('a_norm_F_end', float(rcn_cell.A_dag.detach().norm())):.3f} "
              f"max|A|={m.get('a_max_abs_end', float('nan')):.3f} "
              f"sparsity≈0={100*m.get('a_sparsity_end', float('nan')):.1f}%")
print("[Cell 6] Stage 1 (11-node) trained")


In [ ]:
# >>> Cell 7 : O3 gate + freeze + A_dag freeze verification
# P0-B fix (audit IA) : causal_ablation_check appelle
# iterate_batches_fn(data_loader, builder, device) — 3 arguments positionnels.
# iterate_batches_v6 a exactement cette signature → le passer DIRECTEMENT.
ablation = causal_ablation_check(
    encoder=encoder, rcn_runner=rcn_runner, rcn_cell=rcn_cell,
    regression_head=regression_head, data_loader=val_dataset,
    iterate_batches_fn=iterate_batches_v6,
    builder=builder, device=DEVICE,
    n_samples=int(CONFIG.two_stage.causal_ablation.n_samples),
    threshold=float(CONFIG.two_stage.causal_ablation.threshold))
print(f"[Cell 7] O3 gate ratio={ablation['ratio']:.4f} passes={ablation['passes']}")
if not ablation["passes"]:
    raise RuntimeError("O3 gate FAILED — DAG decorative, abort Stage 2")

freeze_stage1(encoder, rcn_runner.cell, regression_head)
assert not rcn_cell.A_dag.requires_grad, "A_dag NOT frozen"
print("[Cell 7] Stage 1 frozen (encoder + rcn_cell incl. A_dag + regression_head)")

# --- Recalibration sigma_data (safeguard 9-node perdu — revue ML) -----------
# OBLIGATOIRE entre Stage 1 fige et Stage 2 : le mu_HR 11-node change
# std(delta_target = log1p(HR) - log1p(baseline) - mu_HR) vs la valeur du yaml.
# Sans recalibration, le poids de preconditionnement EDM lambda(sigma) est
# mal centre -> Stage 2 sous-optimal (Karras 2022 Eq. 7). deepcopy en Cell 9/10
# heritent de l'edm_config recalibre car faits APRES ce bloc.
from st_cdgm.models.edm_preconditioner import EDMConfig as _EDMConfig
if S1_RESUMED and S1_SIGMA_DATA == S1_SIGMA_DATA:   # resume : sigma deja calcule
    _new_sd = S1_SIGMA_DATA
    print(f"[Cell 7] sigma_data repris du ckpt = {_new_sd:.5f} (pas de recalcul)")
else:
    _calib = calibrate_sigma_data_two_stage(
        encoder=encoder, rcn_runner=rcn_runner, regression_head=regression_head,
        data_loader=train_dataset, iterate_batches_fn=iterate_batches_v6,
        builder=builder, device=DEVICE, max_samples=200)
    _new_sd = float(_calib["sigma_data"])
    print(f"[Cell 7] sigma_data recalibre = {_new_sd:.5f} (mean={_calib.get('mean', float('nan')):+.4f})")
_scale  = float(CONFIG.two_stage.stage2.get("sigma_min_scale_factor", 0.02))
_new_sm = max(1e-4, _new_sd * _scale)
CONFIG.diffusion.edm.sigma_data = _new_sd
CONFIG.diffusion.edm.sigma_min  = _new_sm
diffusion.edm_config = _EDMConfig.from_yaml_dict(CONFIG.diffusion.get("edm", {}))
print(f"[Cell 7] edm sigma_data={_new_sd:.5f} sigma_min={_new_sm:.5f}")

# --- Sauvegarde Stage 1 FINAL (poids figes + A_dag + sigma_data) — protege les ~15h.
if not (SMOKE_MODE or S1_RESUMED):
    _atomic_save({
        "encoder_state_dict": encoder.state_dict(),
        "rcn_cell_state_dict": rcn_cell.state_dict(),
        "regression_head_state_dict": regression_head.state_dict(),
        "A_dag_final": rcn_cell.A_dag.detach().cpu().numpy(),
        "sigma_data": _new_sd, "sigma_min": _new_sm,
        "o3_ratio": float(ablation["ratio"]),
    }, S1_CKPT)
    # progression obsolete une fois le final ecrit (evite un resume partiel a tort)
    if os.path.exists(S1_PROG):
        try: os.remove(S1_PROG)
        except OSError: pass
    print(f"[Cell 7] Stage 1 checkpoint FINAL sauve -> {S1_CKPT}")


In [ ]:
# >>> Cell 8 : BS32b cache WITH lr_fields (full-LR conditioning)
# Persistance (safeguard 9-node) : le cache (mu_HR/baseline/delta/lr_fields sur
# tout le train) coute cher a recalculer — on le sauve/recharge pour survivre a
# une deconnexion Colab entre Stage 1 et Stage 2.
CACHE_PATH = f"{CKPT_DIR}/bs32b_cache_seed42.pt"
if (not SMOKE_MODE) and os.path.exists(CACHE_PATH):
    print(f"[Cell 8] cache present -> chargement {CACHE_PATH}")
    cache = torch.load(CACHE_PATH, map_location="cpu", weights_only=False)
else:
    cache = precompute_stage1_outputs(
        encoder=encoder, rcn_runner=rcn_runner, regression_head=regression_head,
        train_dataset=train_dataset,
        iterate_batches_fn=lambda s: convert_sample_to_batch(s, builder, DEVICE),
        device=DEVICE, dag_variants=["normal"], cache_lr_fields=True)
    if not SMOKE_MODE:
        _atomic_save(cache, CACHE_PATH)
        print(f"[Cell 8] cache sauve -> {CACHE_PATH}")
assert "lr_fields" in cache, "V6' cache MUST contain lr_fields"
print(f"[Cell 8] cache keys = {sorted(cache.keys())}")
print(f"[Cell 8] lr_fields cached shape = {tuple(cache['lr_fields'].shape)} (native res)")

from torch.utils.data import DataLoader as _DL, Dataset as _DS
class _CondDS(_DS):
    def __init__(self, c):
        self.mu=c["mu_HR"]; self.base=c["baseline_log"]; self.delta=c["delta_target"]
        self.mask=c["valid_mask"]; self.lr=c["lr_fields"]
    def __len__(self): return self.mu.shape[0]
    def __getitem__(self, i):
        return {"mu_HR": self.mu[i], "baseline_log": self.base[i], "delta_target": self.delta[i],
                "valid_mask": self.mask[i], "lr_fields": self.lr[i]}
BATCH = 32 if SMOKE_MODE else 48   # 64->48 : plafond VRAM <70GB (+ grad checkpointing)
cached_loader = _DL(_CondDS(cache), batch_size=BATCH, shuffle=True, num_workers=0, drop_last=True)
print(f"[Cell 8] cached_loader ready (batch={BATCH}, {len(cached_loader)} batches)")


In [ ]:
# >>> Cell 9 : SMOKE Stage 2 (optionnel) — deja validee, sautee par defaut
import copy, torch.nn.functional as F, os
# SMOKE deja validee (A2 apparie +6.4%). Inutile une fois le Stage 2 entraine.
# Sautee par defaut ; auto-skip si le checkpoint Stage 2 final existe deja.
# Mettre RUN_SMOKE=True pour la re-executer explicitement.
RUN_SMOKE = False
_stage2_done = os.path.exists(f"{CKPT_DIR}/v6_prime_seed42.pth")
if (not RUN_SMOKE) or _stage2_done:
    print(f"[Cell 9] SMOKE sautee (RUN_SMOKE={RUN_SMOKE}, stage2_done={_stage2_done}) — "
          "deja validee : A2 apparie +6.4%.")
else:
    S9_EPOCHS = 8   # + d'epoques : l'usage du LR par le debruiteur peut emerger tard
    diffusion_smoke = copy.deepcopy(diffusion)
    opt_smoke = torch.optim.AdamW(diffusion_smoke.parameters(), lr=float(CONFIG.two_stage.stage2.lr), weight_decay=1e-4)

    # --- A2 APPARIE : meme bruit on/off -> la difference ne vient QUE du LR.
    def _a2_paired(n=20):
        diffusion_smoke.eval()
        d_on = d_off = 0.0; k = 0
        with torch.no_grad():
            for b in cached_loader:
                mu=b["mu_HR"].to(DEVICE); bl=b["baseline_log"].to(DEVICE); dt=b["delta_target"].to(DEVICE)
                lr=b["lr_fields"].to(DEVICE)
                if lr.shape[-2:] != dt.shape[-2:]:
                    lr = F.interpolate(lr, size=dt.shape[-2:], mode="bilinear", align_corners=False)
                _sd = 4321 + k
                torch.manual_seed(_sd)
                l_on = float(diffusion_smoke.compute_loss_edm(target=dt, mu_HR=mu, baseline_log=bl, lr_fields=lr))
                torch.manual_seed(_sd)   # MEME bruit -> comparaison appariee
                l_off = float(diffusion_smoke.compute_loss_edm(target=dt, mu_HR=mu, baseline_log=bl, lr_fields=torch.zeros_like(lr)))
                d_on += l_on; d_off += l_off; k += 1
                if k >= n: break
        diffusion_smoke.train()
        return d_on/max(1,k), d_off/max(1,k)

    _a2_hist = []
    for ep in range(S9_EPOCHS):
        m = train_epoch_stage2_cached(diffusion_decoder=diffusion_smoke, optimizer=opt_smoke,
            cached_dataloader=cached_loader, device=DEVICE, use_amp=True, gradient_clipping=1.0, log_every=50)
        _lon, _loff = _a2_paired()
        _a2 = 100*(_loff - _lon)/max(1e-6, _lon)
        _a2_hist.append(_a2)
        print(f"SMOKE ep{ep+1}/{S9_EPOCHS} loss_diff={m['loss_diff']:.4f}  "
              f"corr(D_y,mu_HR)={m.get('corr_dy_mu', float('nan')):+.3f}  A2(apparie)={_a2:+.1f}%")
        _c = m.get("corr_dy_mu", float("nan"))
        if _c == _c and _c < -0.3:
            raise RuntimeError(
                f"SMOKE ABORT : corr(D_y, mu_HR) = {_c:.3f} < -0.3 — basculer sur V6'.1.")

    _a2_final = sum(_a2_hist[-3:]) / max(1, len(_a2_hist[-3:]))
    print()
    print(f"[SMOKE A2] historique = {['%+.1f%%'%x for x in _a2_hist]}")
    print(f"[SMOKE A2] moyenne 3 dernieres = {_a2_final:+.1f}%  (>+2% => le debruiteur UTILISE le LR)")
    print("  ✓ A2 POSITIF" if _a2_final > 2.0 else ("  ~ A2 FAIBLE" if _a2_final > 0 else "  ⛔ A2 <= 0"))

    import gc as _gc
    del diffusion_smoke, opt_smoke
    _gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"[Cell 9] VRAM apres nettoyage smoke : {torch.cuda.memory_allocated()/1e9:.1f} GB")


In [ ]:
# >>> Cell 10 : Stage 2 FULL (full-LR conditioning) + EMA + persist
import copy, os
S2_EPOCHS = 5 if SMOKE_MODE else 200
save_dir = CKPT_DIR   # {DRIVE_ROOT}/oracle_v6_prime/seed_42 (defini Cell 6)
FINAL_CKPT = f"{save_dir}/v6_prime_seed42.pth"
S2_CKPT    = f"{save_dir}/stage2_ckpt_seed42.pth"   # checkpoint periodique (resume)

ema = copy.deepcopy(diffusion).eval()
for p in ema.parameters(): p.requires_grad_(False)
opt_s2 = torch.optim.AdamW(diffusion.parameters(), lr=float(CONFIG.two_stage.stage2.lr), weight_decay=1e-4)

# RESUME Stage 2 : reprend a l'epoque sauvee (survie deconnexion Colab).
_start_ep = 0
if (not SMOKE_MODE) and os.path.exists(S2_CKPT):
    _c2 = torch.load(S2_CKPT, map_location=DEVICE, weights_only=False)
    diffusion.load_state_dict(_c2["diffusion_state_dict"])
    ema.load_state_dict(_c2["ema_state_dict"])
    try: opt_s2.load_state_dict(_c2["opt_state_dict"])
    except Exception as _e: print(f"[Cell 10] opt state non repris ({_e})")
    _start_ep = int(_c2.get("epoch_done", 0))
    print(f"[Cell 10] Stage 2 RESUME depuis epoque {_start_ep}/{S2_EPOCHS}")

def _save_stage2(epoch_done, path):
    _atomic_save({"diffusion_state_dict": diffusion.state_dict(), "ema_state_dict": ema.state_dict(),
                  "opt_state_dict": opt_s2.state_dict(), "epoch_done": epoch_done,
                  "encoder_state_dict": encoder.state_dict(), "rcn_cell_state_dict": rcn_cell.state_dict(),
                  "regression_head_state_dict": regression_head.state_dict(),
                  "lr_conditioning_channels": LR_COND_CHANNELS, "stage2_cond_lr_vars": STAGE2_COND_LR_VARS,
                  "sigma_data": float(CONFIG.diffusion.edm.sigma_data),
                  "A_dag_final": rcn_cell.A_dag.detach().cpu().numpy()}, path)

for ep in range(_start_ep, S2_EPOCHS):
    m = train_epoch_stage2_cached(diffusion_decoder=diffusion, optimizer=opt_s2,
        cached_dataloader=cached_loader, device=DEVICE, use_amp=True, gradient_clipping=1.0,
        log_every=50, ema_model=ema, ema_decay=0.9999)
    if (ep+1) % 10 == 0 or ep == 0:
        print(f"[S2 ep{ep+1}/{S2_EPOCHS}] loss_diff={m['loss_diff']:.5f}")
    # Checkpoint APRES CHAQUE epoque (choix user) — ecriture atomique (.tmp+replace).
    if not SMOKE_MODE:
        _save_stage2(ep + 1, S2_CKPT)

_save_stage2(S2_EPOCHS, FINAL_CKPT)   # checkpoint FINAL (consomme par eval/3-way)
print(f"[Cell 10] saved → {FINAL_CKPT}")


In [ ]:
# >>> Cell 11 : Eval FULL TEST SPLIT + ablations A1/A2 — END-TO-END
# P0 fix (audits Recherche + ML) : verdict co-primaire sur le TEST SPLIT
# COMPLET (2 ans), pas 16 echantillons (~0.16 evenement p99/pixel sur 16 jours
# = statistiquement indefini, incomparable aux references).
# Protocole : VERDICT = full split, K=64. ATTRIBUTION A1/A2 = full split, K=16
# (3 conditions au MEME K -> differences comparables, compute maitrise).
import torch.nn.functional as F, numpy as np, json
from st_cdgm.evaluation.eval_metrics_dual_convention import evaluate_ensemble

# Budget éval adapté au GPU (L4 22GB ~3-4x plus lent que A100). K plus petit +
# moins de pas -> tractable. La comparaison 3-way reste valide (memes reglages
# pour V6'/V5/noncausal dans le meme run). Remonter K/NUM_STEPS si A100 dispo.
K_VERDICT   = 8 if SMOKE_MODE else 32
K_ABLATION  = 4 if SMOKE_MODE else 12
NUM_STEPS   = 24
EVAL_BATCH  = 8    # L4 22GB : batch 16 peut faire du thrashing VRAM (crawl) sur
                   # les modeles baseline -> 8 pour rester en VRAM. Monter si A100.

# --- climatology per-pixel thresholds (Convention A, ETCCDI) ---------------
# P0 fix : clim_p95_p99.npz (issu du run phase8/9-node) peut etre absent. On le
# CALCULE alors depuis le HR CIBLE de la fenetre train : HR_vrai = baseline_log +
# mu_HR + delta_target reconstruit exactement le HR (independant du modele), en
# mm/day, puis quantiles per-pixel. Standard ETCCDI (reference = train).
from st_cdgm.evaluation.eval_metrics_dual_convention import to_mm_day as _to_mm_day
_gen_clim = f"{DRIVE_ROOT}/oracle_v6_prime/seed_42/clim_p95_p99.npz"
_cp = next((p for p in (
    f"{DRIVE_ROOT}/oracle_9node/seed_42/phase8/clim_p95_p99.npz",
    f"{DRIVE_ROOT}/ckpt_v2_corrdiff_normal/clim_p95_p99.npz",
    _gen_clim) if Path(p).exists()), None)
if _cp is not None:
    _clim = np.load(_cp)
    clim_p99 = torch.from_numpy(_clim["clim_p99"].astype(np.float32)).to(DEVICE)
    clim_p95 = torch.from_numpy(_clim["clim_p95"].astype(np.float32)).to(DEVICE)
    print(f"[Cell 11] climatology loaded from {_cp}")
else:
    print("[Cell 11] clim absente -> calcul depuis le cache train (HR cible, mm/day)")
    _tc = cache if "cache" in globals() else torch.load(CACHE_PATH, map_location="cpu", weights_only=False)
    with torch.no_grad():
        _hr_log = (_tc["baseline_log"] + _tc["mu_HR"] + _tc["delta_target"]).float()
        _hr_mm = _to_mm_day(_hr_log).squeeze(1).cpu().numpy()   # [N,H,W] mm/day
    _p95 = np.nanpercentile(_hr_mm, 95.0, axis=0).astype(np.float32)
    _p99 = np.nanpercentile(_hr_mm, 99.0, axis=0).astype(np.float32)
    os.makedirs(os.path.dirname(_gen_clim), exist_ok=True)
    np.savez(_gen_clim, clim_p95=_p95, clim_p99=_p99)
    clim_p95 = torch.from_numpy(_p95).to(DEVICE)
    clim_p99 = torch.from_numpy(_p99).to(DEVICE)
    print(f"[Cell 11] climatology CALCULEE sur {_hr_mm.shape[0]} jours train -> {_gen_clim} "
          f"| p99 median={np.nanmedian(_p99):.1f} mm/day p95 median={np.nanmedian(_p95):.1f}")

# --- materialise test conditioning on the FULL split ------------------------
test_cache = precompute_stage1_outputs(
    encoder=encoder, rcn_runner=rcn_runner, regression_head=regression_head,
    train_dataset=test_dataset,
    iterate_batches_fn=lambda s: convert_sample_to_batch(s, builder, DEVICE),
    device=DEVICE, dag_variants=["normal"], cache_lr_fields=True)

N_TOTAL = test_cache["mu_HR"].shape[0]
N_TEST = min(16, N_TOTAL) if SMOKE_MODE else N_TOTAL   # FULL split hors smoke
print(f"[Cell 11] test samples : {N_TEST} / {N_TOTAL}")
mu_all    = test_cache["mu_HR"][:N_TEST]
base_all  = test_cache["baseline_log"][:N_TEST]
delta_all = test_cache["delta_target"][:N_TEST]
lr_all    = test_cache["lr_fields"][:N_TEST]

import time as _time
def sample_ensemble(zero_mu=False, zero_lr=False, K=None, label="verdict"):
    """Batched ensemble sampler avec commutateurs A1/A2 (M3 : lr_fields requis).
    cfg_scale=0.0 (P2-ii audit IA) : semantique conditioned-only identique a
    1.0 sur edm_karras, mais evite tout double-forward CFG. Prints de progression
    par membre (+ ETA) : le sampling L4 est long, il faut voir qu'il avance."""
    if K is None: K = K_VERDICT
    ema.eval()
    members = []
    _t0 = _time.time()
    with torch.no_grad():
        for k in range(K):
            torch.manual_seed(1000 + k)
            chunks = []
            for i0 in range(0, N_TEST, EVAL_BATCH):
                sl = slice(i0, min(i0 + EVAL_BATCH, N_TEST))
                mu_ = mu_all[sl].to(DEVICE); bl_ = base_all[sl].to(DEVICE)
                lr_ = lr_all[sl].to(DEVICE)
                if lr_.shape[-2:] != mu_.shape[-2:]:
                    lr_ = F.interpolate(lr_, size=mu_.shape[-2:], mode="bilinear", align_corners=False)
                if zero_mu: mu_ = torch.zeros_like(mu_)
                if zero_lr: lr_ = torch.zeros_like(lr_)   # A2 = zero post-norm
                o = ema.sample(conditioning=None, num_steps=NUM_STEPS,
                               scheduler_type="edm_karras", cfg_scale=0.0,
                               mu_HR=mu_, baseline_log=bl_, lr_fields=lr_)
                chunks.append(o.residual.cpu())
            members.append(torch.cat(chunks, dim=0))
            if (k + 1) % 2 == 0 or k == 0:
                _el = _time.time() - _t0
                _eta = _el / (k + 1) * (K - k - 1)
                print(f"    [{label}] membre {k+1}/{K} | {_el:.0f}s ecoule | ETA {_eta:.0f}s", flush=True)
    return torch.stack(members, 0)   # [K, N_TEST, 1, H, W] sur CPU

mu_t, base_t, delta_t = mu_all.to(DEVICE), base_all.to(DEVICE), delta_all.to(DEVICE)
def M(ens):
    return evaluate_ensemble(ens.to(DEVICE), mu_t, base_t, delta_t, clim_p99, clim_p95)

# PERSISTANCE (demande user) : le sampling V6' coute ~8h sur L4. On sauve les
# resultats (dicts de metriques) et on les RECHARGE au prochain run -> plus jamais
# de re-sampling V6'. Supprimer le .json pour forcer un recalcul.
V6_EVAL_JSON = f"{CKPT_DIR}/v6_prime_eval_results.json"
if (not SMOKE_MODE) and os.path.exists(V6_EVAL_JSON):
    with open(V6_EVAL_JSON) as _f: _sv = json.load(_f)
    res_full, res_refA = _sv["res_full"], _sv["res_refA"]
    res_a1, res_a2 = _sv["res_a1"], _sv["res_a2"]
    print(f"[Cell 11] resultats V6' RECHARGES depuis {V6_EVAL_JSON} (pas de re-sampling)")
else:
    print(f"[Cell 11] VERDICT sampling (full split, K={K_VERDICT}) ...")
    res_full = M(sample_ensemble(K=K_VERDICT, label="verdict"))
    print(f"[Cell 11] ATTRIBUTION sampling (K={K_ABLATION} x 3 conditions) ...")
    res_refA  = M(sample_ensemble(K=K_ABLATION, label="attr-ref"))
    res_a1    = M(sample_ensemble(zero_mu=True, K=K_ABLATION, label="A1(mu=0)"))
    res_a2    = M(sample_ensemble(zero_lr=True, K=K_ABLATION, label="A2(lr=0)"))
    if not SMOKE_MODE:
        _sv = {k: {kk: float(vv) for kk, vv in d.items()}
               for k, d in [("res_full", res_full), ("res_refA", res_refA),
                            ("res_a1", res_a1), ("res_a2", res_a2)]}
        with open(V6_EVAL_JSON, "w") as _f: json.dump(_sv, _f, indent=2)
        print(f"[Cell 11] resultats V6' SAUVES -> {V6_EVAL_JSON}")

results = {
    "F1_p99_pergrid": res_full["conv_A_F1p99"],
    "F1_p99_pooled":  res_full["conv_B_F1p99"],
    "RMSE": res_full["rmse"], "Pearson": res_full["pearson_global"],
    "Rx1day_bias": res_full["rx1day_bias"], "RAPSD": res_full["rapsd_distance"],
    "n_test": int(N_TEST), "K_verdict": int(K_VERDICT), "K_ablation": int(K_ABLATION),
    # Attribution : triple au MEME K (comparabilite interne, semantique A1
    # INFORMATIONNELLE : conditioning ablate, mu reel conserve en recomposition
    # — pre-declare dans le prereg)
    "ref_KA_F1_pooled":    res_refA["conv_B_F1p99"],
    "A1_mu_off_F1_pooled": res_a1["conv_B_F1p99"],
    "A2_lr_off_F1_pooled": res_a2["conv_B_F1p99"],
    "ref_KA_F1_pergrid":    res_refA["conv_A_F1p99"],
    "A1_mu_off_F1_pergrid": res_a1["conv_A_F1p99"],
    "A2_lr_off_F1_pergrid": res_a2["conv_A_F1p99"],
}
print(); print("=== V6' RESULTS (full test split) ===")
for k, v in results.items(): print(f"  {k:26s} = {v}")
print(); print(f"  A1 (causal) contribution pooled  = {results['ref_KA_F1_pooled']-results['A1_mu_off_F1_pooled']:+.4f}")
print(f"  A2 (full-LR) contribution pooled = {results['ref_KA_F1_pooled']-results['A2_lr_off_F1_pooled']:+.4f}")


In [ ]:
# >>> Cell 12 : Verdict vs seuils pre-enregistres + gate OOD
import json, os
seuils = json.load(open("path_c_plus/audit/V6_PRIME_seuils_preregistered.json"))
thr_pg = seuils["targets_to_beat"]["co_primary_1_per_gridpoint"]
thr_pl = seuils["targets_to_beat"]["co_primary_2_pooled"]

# --- P0 (audit Recherche) : les references per-gridpoint 0.841/0.816 ont ete
# calculees avec l'ANCIENNE metrique Conv A (quantile scalaire, buggee).
# Elles doivent etre RECOMPUTEES avec la metrique corrigee (broadcast per-pixel)
# via _eval_3way_dual_convention re-execute. Ce notebook cherche le fichier de
# references recomputees ; sinon le verdict per-gridpoint est marque STALE.
RECOMPUTED_REFS = f"{DRIVE_ROOT}/oracle_v6_prime/recomputed_pergrid_references.json"
if Path(RECOMPUTED_REFS).exists():
    _refs = json.load(open(RECOMPUTED_REFS))
    ref_pg_noncausal = float(_refs["noncausal_v4_F1p99_pergrid"])
    ref_pg_v5        = float(_refs["v5_causal_F1p99_pergrid"])
    refs_status = "RECOMPUTED"
else:
    ref_pg_noncausal = float(thr_pg["noncausal_v4"])
    ref_pg_v5        = float(thr_pg["v5_causal_seed42"])
    refs_status = "STALE_OLD_METRIC"
    print("  !! References per-gridpoint NON recomputees avec la metrique corrigee")
    print("  !! -> verdict co-primaire 1 = provisoire. Re-executer le 3-way eval")
    print(f"  !! et sauvegarder {RECOMPUTED_REFS}")

pg, pl = results["F1_p99_pergrid"], results["F1_p99_pooled"]
v_pg = "PASS" if (pg >= ref_pg_noncausal and pg >= ref_pg_v5) else "FAIL"
if refs_status != "RECOMPUTED":
    v_pg = f"{v_pg}_PROVISOIRE_REFS_STALE"
v_pl = ("PASS_STRONG" if pl >= thr_pl["PASS_STRONG"] else "PASS_TARGET" if pl >= thr_pl["PASS_TARGET"]
        else "PASS_MINIMAL" if pl >= thr_pl["PASS_MINIMAL"] else "FAIL")

# --- Gate OOD (P1 audit Recherche + fix ML) : le verdict n'est FINAL qu'avec
# l'OOD EC-Earth3 complete. Le run OOD = relancer CE notebook avec
# GCM_ID="EC-Earth3" (Cell 2) — il ecrira son propre verdict ; ce champ trace.
OOD_VERDICT_PATH = f"{DRIVE_ROOT}/oracle_v6_prime/seed_42/v6_prime_verdict_EC-Earth3.json"
ood_status = "DONE" if Path(OOD_VERDICT_PATH).exists() else "PENDING"

verdict = {
    "gcm": GCM_ID,
    "per_gridpoint": v_pg, "pooled": v_pl,
    "pergrid_refs_status": refs_status,
    "at_least_one_coprimary": (v_pg.startswith("PASS") or v_pl != "FAIL"),
    "ood_EC-Earth3": ood_status if not IS_OOD_RUN else "THIS_IS_THE_OOD_RUN",
    "final": (ood_status == "DONE" and refs_status == "RECOMPUTED") if not IS_OOD_RUN else True,
    "results": results,
}
print("=== V6' VERDICT ===")
print(f"  per-gridpoint : {v_pg}  (F1={pg:.4f} vs noncausal {ref_pg_noncausal} / V5 {ref_pg_v5} [{refs_status}])")
print(f"  pooled        : {v_pl}  (F1={pl:.4f} vs target {thr_pl['PASS_TARGET']})")
print(f"  OOD EC-Earth3 : {verdict['ood_EC-Earth3']}")
print(f"  FINAL         : {verdict['final']}  (exige OOD DONE + refs RECOMPUTED)")

os.makedirs(f"{DRIVE_ROOT}/oracle_v6_prime/seed_42", exist_ok=True)
_out = (f"{DRIVE_ROOT}/oracle_v6_prime/seed_42/v6_prime_verdict_{GCM_ID}.json"
        if IS_OOD_RUN else f"{DRIVE_ROOT}/oracle_v6_prime/seed_42/v6_prime_verdict.json")
json.dump(verdict, open(_out, "w"), indent=2)
print(f"[Cell 12] verdict saved -> {_out}")


In [ ]:
# >>> Cell 13 : 3-WAY — BUILD des baselines (V5_causal + noncausal_v4)
# Re-evalue les DEUX baselines avec la metrique Conv A CORRIGEE, sur le MEME
# test split, meme K, meme composition mm -> les references per-gridpoint
# sont recomputees IN-PLACE (resout le probleme REFS_STALE du prereg).
import torch, numpy as np
from omegaconf import OmegaConf as _OC13
from st_cdgm.models.intelligible_encoder import IntelligibleVariableEncoder, IntelligibleVariableConfig
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder
from st_cdgm.models.regression_mean_predictor import RegressionMeanPredictor, RegressionPredictorConfig
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from st_cdgm.v6_constants import NONCAUSAL_15_VARS

V5_CAUSAL_CKPT = f"{DRIVE_ROOT}/ckpt_v2_corrdiff_normal/epoch_last.pth"
NONCAUSAL_CKPT = f"{DRIVE_ROOT}/ckpt_noncausal/epoch_last.pth"

def _persist_load_state_dict(module, sd, strict=True):
    if sd is None:
        raise RuntimeError(f"state_dict manquant pour {type(module).__name__}")
    missing, unexpected = module.load_state_dict(sd, strict=False)
    if strict and missing:
        _crit = [k for k in missing if "_state_adapter" not in k]
        if _crit:
            raise RuntimeError(f"missing keys: {_crit[:6]}")
    return module

def _sd_tensor(sd, key):
    return sd.get(key) if sd else None

def _infer_unet_arch_from_sd(sd):
    conv_w = _sd_tensor(sd, "unet.conv_in.weight")
    proj_w = _sd_tensor(sd, "unet.class_embedding.linear_1.weight")
    causal_concat = bool(conv_w is not None and int(conv_w.shape[1]) == 3)
    proj_dim = int(proj_w.shape[1]) if proj_w is not None else None
    return causal_concat, proj_dim

# --- CONFIG baselines : base + corrdiff_normal, 15 vars, normalisation 15-var
CONFIG_B = _OC13.load("config/training_config.yaml")
CONFIG_B = _OC13.merge(CONFIG_B, _OC13.load("config/training_config_corrdiff_normal.yaml"))
_OC13.set_struct(CONFIG_B, False)
CONFIG_B.data.lr_variables = list(NONCAUSAL_15_VARS)

MEANS_15 = f"{DRIVE_ROOT}/train/means_ACCESS-CM2.nc"
STDS_15  = f"{DRIVE_ROOT}/train/stds_ACCESS-CM2.nc"
pipeline_15 = NetCDFDataPipeline(
    lr_path=LR_PATH_V6, hr_path=HR_PATH,
    static_path=STATIC_PATH if Path(STATIC_PATH).exists() else None,
    seq_len=int(CONFIG_B.data.seq_len),
    baseline_strategy=str(CONFIG_B.data.baseline_strategy),
    baseline_factor=int(CONFIG_B.data.baseline_factor),
    normalize=bool(CONFIG_B.data.normalize),
    nan_fill_strategy=str(CONFIG_B.data.nan_fill_strategy),
    precipitation_delta=float(CONFIG_B.data.precipitation_delta),
    lr_variables=list(NONCAUSAL_15_VARS),
    hr_variables=list(CONFIG_B.data.hr_variables),
    static_variables=list(CONFIG_B.data.static_variables) if CONFIG_B.data.get("static_variables") else [],
    means_path=MEANS_15 if Path(MEANS_15).exists() else None,
    stds_path=STDS_15 if Path(STDS_15).exists() else None,
    train_start_date=K9_DATES["train"][0], train_end_date=K9_DATES["train"][1],
    val_start_date=K9_DATES["val"][0],     val_end_date=K9_DATES["val"][1],
    test_start_date=K9_DATES["test"][0],   test_end_date=K9_DATES["test"][1],
    temporal_holdout_start_date=K9_DATES["holdout"][0],
    temporal_holdout_end_date=K9_DATES["holdout"][1],
)
builder_15 = HeteroGraphBuilder(
    lr_shape=tuple(CONFIG_B.graph.lr_shape), hr_shape=tuple(CONFIG_B.graph.hr_shape),
    static_dataset=pipeline_15.get_static_dataset(), include_mid_layer=True,
    extended_9node=False, extended_v6_wind=False)   # baselines = ere 6-node
test_15 = pipeline_15.build_sequence_dataset(split="test", seq_len=int(CONFIG_B.data.seq_len),
    stride=int(CONFIG_B.data.stride), as_torch=True)

# --- V5_causal stack ---------------------------------------------------------
_allowed_b = set(builder_15.dynamic_node_types) | set(builder_15.static_node_types)
_enc_cfgs_b = [IntelligibleVariableConfig(name=m.name, meta_path=(m.src, m.relation, m.target),
                                          pool=m.get("pool", "mean"))
               for m in CONFIG_B.encoder.metapaths if m.src in _allowed_b and m.target in _allowed_b]
if pipeline_15.get_static_dataset() is not None:
    _enc_cfgs_b.append(IntelligibleVariableConfig(name="static",
        meta_path=("SP_HR", "causes", "GP850"), pool="mean"))
v5_encoder = IntelligibleVariableEncoder(configs=_enc_cfgs_b,
    hidden_dim=int(CONFIG_B.encoder.hidden_dim),
    conditioning_dim=int(CONFIG_B.encoder.conditioning_dim)).to(DEVICE)
_nvars_b = len(_enc_cfgs_b)
_probe_b = builder_15.lr_grid_to_nodes(torch.zeros(len(NONCAUSAL_15_VARS), *tuple(CONFIG_B.graph.lr_shape)))
v5_rcn_cell = RCNCell(num_vars=_nvars_b, hidden_dim=int(CONFIG_B.rcn.hidden_dim),
    driver_dim=_probe_b.shape[-1], reconstruction_dim=_probe_b.shape[-1],
    dropout=float(CONFIG_B.rcn.dropout)).to(DEVICE)
v5_rcn_runner = RCNSequenceRunner(v5_rcn_cell, detach_interval=CONFIG_B.rcn.get("detach_interval"))
_rh_b = CONFIG_B.two_stage.regression_head
v5_regression_head = GraphToGridDecoder(d_model=int(_rh_b.d_model),
    hr_h=int(CONFIG_B.diffusion.height), hr_w=int(CONFIG_B.diffusion.width),
    intermediate_h=int(_rh_b.intermediate_h), intermediate_w=int(_rh_b.intermediate_w),
    n_heads=int(_rh_b.n_heads), refine_channels=int(_rh_b.refine_channels),
    output_channels=1).to(DEVICE)

print(f"[Cell 13] Loading V5_causal : {V5_CAUSAL_CKPT}")
_ck_v5 = torch.load(V5_CAUSAL_CKPT, map_location=DEVICE, weights_only=False)
_persist_load_state_dict(v5_encoder, _ck_v5.get("encoder_state_dict"))
_persist_load_state_dict(v5_rcn_cell, _ck_v5.get("rcn_cell_state_dict"))
_persist_load_state_dict(v5_regression_head, _ck_v5.get("regression_head_state_dict"))

_v5_sd = _ck_v5.get("diffusion_ema_state_dict") or _ck_v5.get("ema_state_dict") or _ck_v5.get("diffusion_state_dict")
_v5_cc, _v5_proj = _infer_unet_arch_from_sd(_v5_sd)
_uk_b = _OC13.to_container(CONFIG_B.diffusion.unet_kwargs, resolve=True)
for _k in ("down_block_types", "up_block_types"):
    if _k in _uk_b and isinstance(_uk_b[_k], list): _uk_b[_k] = tuple(_uk_b[_k])
if _v5_proj is not None: _uk_b["projection_class_embeddings_input_dim"] = _v5_proj
v5_diffusion = CausalDiffusionDecoder(
    in_channels=1, conditioning_dim=int(CONFIG_B.diffusion.conditioning_dim),
    height=int(CONFIG_B.diffusion.height), width=int(CONFIG_B.diffusion.width),
    unet_kwargs=_uk_b, scheduler_type="edm_karras",
    edm_config=EDMConfig.from_yaml_dict(CONFIG_B.diffusion.get("edm", {})),
    causal_concat=_v5_cc).to(DEVICE)
_persist_load_state_dict(v5_diffusion, _v5_sd)
for _m in (v5_encoder, v5_rcn_cell, v5_regression_head, v5_diffusion):
    _m.eval()
    for _p in _m.parameters(): _p.requires_grad_(False)
print(f"[Cell 13] V5_causal loaded (causal_concat={_v5_cc})")

# --- noncausal_v4 stack ------------------------------------------------------
print(f"[Cell 13] Loading noncausal_v4 : {NONCAUSAL_CKPT}")
_ck_nc = torch.load(NONCAUSAL_CKPT, map_location=DEVICE, weights_only=False)
_nc_sd = _ck_nc.get("diffusion_ema_state_dict") or _ck_nc.get("ema_state_dict") or _ck_nc.get("diffusion_state_dict")
_nc_cc, _nc_proj = _infer_unet_arch_from_sd(_nc_sd)
_uk_nc = _OC13.to_container(CONFIG_B.diffusion.unet_kwargs, resolve=True)
for _k in ("down_block_types", "up_block_types"):
    if _k in _uk_nc and isinstance(_uk_nc[_k], list): _uk_nc[_k] = tuple(_uk_nc[_k])
if _nc_proj is not None: _uk_nc["projection_class_embeddings_input_dim"] = _nc_proj
nc_diffusion = CausalDiffusionDecoder(
    in_channels=1, conditioning_dim=int(CONFIG_B.diffusion.conditioning_dim),
    height=int(CONFIG_B.diffusion.height), width=int(CONFIG_B.diffusion.width),
    unet_kwargs=_uk_nc, scheduler_type="edm_karras",
    edm_config=EDMConfig.from_yaml_dict(CONFIG_B.diffusion.get("edm", {})),
    causal_concat=_nc_cc).to(DEVICE)
_persist_load_state_dict(nc_diffusion, _nc_sd)

# noncausal Stage 1 head = RegressionMeanPredictor (mu direct depuis lr_grid)
nc_regression_head = RegressionMeanPredictor(
    RegressionPredictorConfig(
        in_channels=len(NONCAUSAL_15_VARS), out_channels=1,
        lr_height=int(CONFIG_B.graph.lr_shape[0]), lr_width=int(CONFIG_B.graph.lr_shape[1]),
        hr_height=int(CONFIG_B.diffusion.height), hr_width=int(CONFIG_B.diffusion.width),
        block_out_channels=(64, 128, 192), layers_per_block=2, norm_num_groups=16,
        state_adapter_hidden_dim=int(CONFIG_B.rcn.hidden_dim))).to(DEVICE)
_rh_nc_sd = _ck_nc.get("regression_head_state_dict")
_rh_strict = any("_state_adapter." in k for k in (_rh_nc_sd or {}).keys())
_persist_load_state_dict(nc_regression_head, _rh_nc_sd, strict=_rh_strict)
nc_diffusion.eval(); nc_regression_head.eval()
for _p in list(nc_diffusion.parameters()) + list(nc_regression_head.parameters()):
    _p.requires_grad_(False)
print(f"[Cell 13] noncausal_v4 loaded (causal_concat={_nc_cc})")


In [ ]:
# >>> Cell 14 : 3-WAY — SAMPLE + EVAL des baselines (metrique corrigee)
# Meme protocole que V6-prime (Cell 11) : full test split, K_VERDICT membres,
# cfg 0.0 (conditioned-only), composition mm par membre, Conv A per-pixel.
import time
from st_cdgm.evaluation.two_stage_inference import build_two_stage_inputs
from st_cdgm.evaluation.eval_metrics_dual_convention import evaluate_ensemble

K_BASELINE = 8 if SMOKE_MODE else K_VERDICT   # meme K que le verdict V6-prime

# convert 15-var samples (baseline builder, pas de lr_grid V6 requis)
def convert_sample_to_batch_15(sample):
    lr_seq = sample["lr"]
    seq_len = lr_seq.shape[0]
    lr_nodes = [builder_15.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes, dim=0)
    lr0 = lr_nodes[0]
    dyn = {nt: lr0 for nt in builder_15.dynamic_node_types}
    hetero = builder_15.prepare_step_data(dyn).to(DEVICE)
    return {"lr": lr_tensor, "lr_grid": lr_seq, "residual": sample["residual"],
            "baseline": sample.get("baseline"), "hetero": hetero, "time": sample.get("time")}

def eval_baseline(name, variant, regression_head, encoder=None, rcn_runner=None,
                  diffusion=None, causal_concat=True, K=None):
    if K is None: K = K_BASELINE
    core = getattr(diffusion, "_orig_mod", diffusion)
    core = getattr(core, "module", core)
    mus, bls, deltas = [], [], []
    n_done = 0
    t0 = time.time()
    for _sample in test_15:
        if SMOKE_MODE and n_done >= 16: break
        _batch = convert_sample_to_batch_15(_sample)
        _cond, _mu, _bl, _tgt = build_two_stage_inputs(
            _batch, variant=variant, regression_head=regression_head,
            encoder=encoder, rcn_runner=rcn_runner, builder=builder_15, device=DEVICE)
        _delta = torch.nan_to_num(_tgt - _mu, nan=0.0, posinf=0.0, neginf=0.0)
        mus.append(_mu.cpu()); bls.append(_bl.cpu()); deltas.append(_delta.cpu())
        n_done += 1
    mu_c = torch.cat(mus, 0); bl_c = torch.cat(bls, 0); dl_c = torch.cat(deltas, 0)
    N = mu_c.shape[0]
    print(f"  [{name}] {N} test samples, sampling K={K} ...")
    member_list = []
    with torch.no_grad():
        for k in range(K):
            torch.manual_seed(1000 + k)
            chunks = []
            for i0 in range(0, N, EVAL_BATCH):
                sl = slice(i0, min(i0 + EVAL_BATCH, N))
                kw = dict(num_steps=NUM_STEPS, scheduler_type="edm_karras",
                          cfg_scale=0.0, apply_constraints=False)
                if causal_concat:
                    kw["mu_HR"] = mu_c[sl].to(DEVICE)
                    kw["baseline_log"] = bl_c[sl].to(DEVICE)
                o = core.sample(conditioning=None, **kw)
                chunks.append(o.residual.cpu())
            member_list.append(torch.cat(chunks, 0))
            if (k + 1) % 2 == 0 or k == 0:   # visibilite : print par membre (etait %16)
                _el = time.time() - t0
                _eta = _el / (k + 1) * (K - k - 1)
                print(f"    [{name}] membre {k+1}/{K} | {_el:.0f}s ecoule | ETA {_eta:.0f}s", flush=True)
    ens = torch.stack(member_list, 0)
    res = evaluate_ensemble(ens.to(DEVICE), mu_c.to(DEVICE), bl_c.to(DEVICE),
                            dl_c.to(DEVICE), clim_p99, clim_p95)
    print(f"  [{name}] conv_A F1p99={res['conv_A_F1p99']:.4f}  conv_B F1p99={res['conv_B_F1p99']:.4f}  RMSE={res['rmse']:.4f}")
    return res

print("[Cell 14] === V5_causal (metrique corrigee) ===")
res_v5 = eval_baseline("V5_causal", "causal", v5_regression_head,
                        encoder=v5_encoder, rcn_runner=v5_rcn_runner,
                        diffusion=v5_diffusion, causal_concat=_v5_cc)
print("[Cell 14] === noncausal_v4 (metrique corrigee) ===")
res_nc = eval_baseline("noncausal_v4", "noncausal", nc_regression_head,
                        diffusion=nc_diffusion, causal_concat=_nc_cc)


In [ ]:
# >>> Cell 15 : 3-WAY TABLE + references recomputees + VERDICT FINAL
import json, os

table_metrics = ["conv_A_F1p99", "conv_A_F1p95", "conv_B_F1p99", "conv_B_F1p95",
                 "rmse", "mae", "pearson_global", "rapsd_distance", "rx1day_bias"]
rows = {"V6_prime": res_full, "V5_causal": res_v5, "noncausal_v4": res_nc}

print("=" * 88)
print("3-WAY COMPARISON (Conv A CORRIGEE per-pixel, composition mm, full test split)")
print("=" * 88)
hdr = f"{'Metric':24s} " + " ".join(f"{n:>14s}" for n in rows)
print(hdr)
lower_better = {"rmse", "mae", "rapsd_distance"}
for met in table_metrics:
    vals = {n: rows[n].get(met, float("nan")) for n in rows}
    _finite = [n for n in vals if vals[n] == vals[n]]
    best = (min if met in lower_better else max)(_finite, key=lambda n: vals[n]) if _finite else None
    line = f"{met:24s} " + " ".join(f"{vals[n]:14.4f}" for n in rows)
    print(line + (f"   <- best: {best}" if best else ""))

# --- Save RECOMPUTED per-gridpoint references (resout REFS_STALE) -----------
os.makedirs(f"{DRIVE_ROOT}/oracle_v6_prime", exist_ok=True)
recomputed = {
    "computed_with": "Conv A per-pixel ETCCDI corrigee + composition mm par membre",
    "protocol": {"n_test": int(results.get("n_test", 0)),
                  "K": int(K_BASELINE), "num_steps": int(NUM_STEPS), "gcm": GCM_ID},
    "noncausal_v4_F1p99_pergrid": float(res_nc["conv_A_F1p99"]),
    "v5_causal_F1p99_pergrid":    float(res_v5["conv_A_F1p99"]),
    "noncausal_v4_F1p99_pooled":  float(res_nc["conv_B_F1p99"]),
    "v5_causal_F1p99_pooled":     float(res_v5["conv_B_F1p99"]),
}
REFS_OUT = f"{DRIVE_ROOT}/oracle_v6_prime/recomputed_pergrid_references.json"
json.dump(recomputed, open(REFS_OUT, "w"), indent=2)
print(); print(f"[Cell 15] references recomputees sauvegardees -> {REFS_OUT}")

# --- VERDICT FINAL (references recomputees, in-protocol) ---------------------
pg, pl = results["F1_p99_pergrid"], results["F1_p99_pooled"]
ref_pg_nc, ref_pg_v5 = recomputed["noncausal_v4_F1p99_pergrid"], recomputed["v5_causal_F1p99_pergrid"]
ref_pl_nc = recomputed["noncausal_v4_F1p99_pooled"]
v_pg_final = "PASS" if (pg >= ref_pg_nc and pg >= ref_pg_v5) else "FAIL"
v_pl_final = "PASS" if pl >= ref_pl_nc else ("PASS_MINIMAL" if pl >= 0.480 else "FAIL")

final = {
    "gcm": GCM_ID,
    "co_primary_1_pergrid": {"v6": pg, "ref_noncausal": ref_pg_nc, "ref_v5": ref_pg_v5, "verdict": v_pg_final},
    "co_primary_2_pooled":  {"v6": pl, "ref_noncausal": ref_pl_nc, "verdict": v_pl_final},
    "at_least_one_coprimary": (v_pg_final == "PASS" or v_pl_final.startswith("PASS")),
    "refs_status": "RECOMPUTED_IN_PROTOCOL",
    "ood_EC-Earth3": "PENDING" if not IS_OOD_RUN else "THIS_IS_THE_OOD_RUN",
    "three_way": {n: {m: float(rows[n].get(m, float("nan"))) for m in table_metrics} for n in rows},
    "v6_ablations": {k: results[k] for k in results if str(k).startswith(("A1_", "A2_", "ref_KA"))},
}
print(); print("=== VERDICT FINAL (references recomputees) ===")
print(f"  co-primaire 1 per-gridpoint : {v_pg_final}  (V6={pg:.4f} vs noncausal={ref_pg_nc:.4f} / V5={ref_pg_v5:.4f})")
print(f"  co-primaire 2 pooled        : {v_pl_final}  (V6={pl:.4f} vs noncausal={ref_pl_nc:.4f})")
print(f"  >>> au moins un co-primaire : {'PASS' if final['at_least_one_coprimary'] else 'FAIL'}")
print(f"  OOD EC-Earth3 : {final['ood_EC-Earth3']} (verdict definitif exige le run OOD)")

_fout = f"{DRIVE_ROOT}/oracle_v6_prime/seed_42/v6_prime_3way_final_{GCM_ID}.json"
os.makedirs(os.path.dirname(_fout), exist_ok=True)
json.dump(final, open(_fout, "w"), indent=2)
print(f"[Cell 15] verdict final 3-way sauvegarde -> {_fout}")
